<a href="https://colab.research.google.com/github/1pawn0/time-series-forecasting-lab/blob/main/sklearn_ensemble_HistGradientBoostingRegressor_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
%pip install -qU polars scikit-learn

In [10]:
import os
from datetime import datetime
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import polars as pl
from google.colab import userdata
from scipy.stats import randint, uniform
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, RobustScaler, SplineTransformer


In [11]:
from urllib.request import urlretrieve
from pathlib import Path

csv_url: str = "https://www.cryptodatadownload.com/cdd/Gemini_BTCUSD_1h.csv"
data_dir: Path = Path("./data")
data_dir.mkdir(exist_ok=True, parents=True)
csv_file: Path = data_dir / "Gemini_BTCUSD_1h.csv"

if not csv_file.exists():
    urlretrieve(csv_url, csv_file)
    print(f"CSV file downloaded to {csv_file}")
else:
    print(f"CSV file already exists at {csv_file}")

import polars as pl

# Define the schema of df
df_schema: dict = {
    "date": pl.Datetime("ms"),
    "close": pl.Float64,
}
# Define which columns to load from the CSV
cols = list(df_schema.keys())
# Read the CSV file
df = (
    pl.read_csv(csv_file, columns=cols, skip_lines=1, schema_overrides=df_schema)
    .sort("date")
    .rename({"close": "price"})
)
# Fill the date gaps inside `df`
full_date_range = pl.datetime_range(
    start=df["date"][0],
    end=df["date"][-1],
    interval="1h",
    time_unit="ms",
    eager=True,
).to_frame(name="date")
df = full_date_range.join(df, on="date", how="left").interpolate().sort("date")
# Filter the dataset to only include data after a specific start date
df = df.filter(pl.col("date") > datetime(2016, 10, 30)).sort("date")
price_pct_changes_df = df.with_columns(
    pl.col("price").pct_change().shift(-1).alias("price_pct_change")
).sort("date")[:-1]
price_pct_changes_df

CSV file already exists at data/Gemini_BTCUSD_1h.csv


date,price,price_pct_change
datetime[ms],f64,f64
2016-10-30 01:00:00,695.23,0.014973
2016-10-30 02:00:00,705.64,0.003713
2016-10-30 03:00:00,708.26,-0.001398
2016-10-30 04:00:00,707.27,-0.000014
2016-10-30 05:00:00,707.26,0.0
…,…,…
2025-09-22 18:00:00,112509.66,-0.002955
2025-09-22 19:00:00,112177.25,0.006117
2025-09-22 20:00:00,112863.47,0.00171


In [12]:
prices = price_pct_changes_df.with_columns([
    pl.col("date").dt.year().alias("year"),
    pl.col("date").dt.month().alias("month"),
    pl.col("date").dt.day().alias("day"),
    pl.col("date").dt.hour().alias("hour"),
    pl.col("date").dt.weekday().alias("weekday"),
    pl.col("date").dt.week().alias("week"),
    pl.col("date").dt.quarter().alias("quarter"),
    pl.col("date").dt.ordinal_day().alias("ordinal_day"),
]).sort('date')


In [13]:
hours: int = 24
lag_expressions: list[pl.Expr] = [pl.col("price_pct_change").shift(i).alias(f"pct_change_lag_{i}h") for i in range(1, hours + 1)]
rolling_expressions: list[pl.Expr] = [
    pl.col('price_pct_change').shift().rolling_mean(hours).alias(f'rolling_mean_{hours}h'),
    pl.col('price_pct_change').shift().rolling_std(hours).alias(f'rolling_std_{hours}h'),
    pl.col('price_pct_change').shift().rolling_var(hours).alias(f'rolling_var_{hours}h'),
    pl.col('price_pct_change').shift().rolling_skew(hours).alias(f'rolling_skew_{hours}h'),
    pl.col('price_pct_change').shift().rolling_kurtosis(hours).alias(f'rolling_kurtosis_{hours}h'),
    pl.col('price_pct_change').shift().ewm_mean_by(by='date', half_life='24h').alias(f'ewm_mean_{hours}h')
]
expr_list = lag_expressions + rolling_expressions

lagged_df = prices.with_columns(expr_list).drop_nulls().drop('price')

In [14]:
cyclical_periods = {
    'month': 12,
    'day': 31,
    'hour': 24,
    'weekday': 7,
    'week': 53,
    'quarter': 4,
    'ordinal_day': 366,
}

cols_to_process = [col for col in cyclical_periods if col in lagged_df.columns]

expressions = []
for col_name in cols_to_process:
    max_val = cyclical_periods[col_name]
    col_expr = pl.col(col_name)

    expressions.append(
        (col_expr * (2 * np.pi / max_val)).sin().alias(f"{col_name}_sine")
    )
    expressions.append(
        (col_expr * (2 * np.pi / max_val)).cos().alias(f"{col_name}_cosine")
    )

processed_df = lagged_df.with_columns(expressions).drop(cols_to_process)

processed_df

date,price_pct_change,year,pct_change_lag_1h,pct_change_lag_2h,pct_change_lag_3h,pct_change_lag_4h,pct_change_lag_5h,pct_change_lag_6h,pct_change_lag_7h,pct_change_lag_8h,pct_change_lag_9h,pct_change_lag_10h,pct_change_lag_11h,pct_change_lag_12h,pct_change_lag_13h,pct_change_lag_14h,pct_change_lag_15h,pct_change_lag_16h,pct_change_lag_17h,pct_change_lag_18h,pct_change_lag_19h,pct_change_lag_20h,pct_change_lag_21h,pct_change_lag_22h,pct_change_lag_23h,pct_change_lag_24h,rolling_mean_24h,rolling_std_24h,rolling_var_24h,rolling_skew_24h,rolling_kurtosis_24h,ewm_mean_24h,month_sine,month_cosine,day_sine,day_cosine,hour_sine,hour_cosine,weekday_sine,weekday_cosine,week_sine,week_cosine,quarter_sine,quarter_cosine,ordinal_day_sine,ordinal_day_cosine
datetime[ms],f64,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2016-10-31 01:00:00,0.013736,2016,-0.018515,0.001733,-0.002529,-0.005653,0.005327,-0.010023,-0.00182,-0.000014,-0.001831,0.006222,0.000567,-0.007013,0.004072,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.000014,-0.001398,0.003713,0.014973,-0.000508,0.006108,0.000037,-0.530966,2.7769,0.006934,-0.866025,0.5,-2.4493e-16,1.0,0.258819,0.965926,0.781831,0.62349,-0.875735,0.482792,-2.4493e-16,1.0,-0.866025,0.5
2016-10-31 02:00:00,-0.00115,2016,0.013736,-0.018515,0.001733,-0.002529,-0.005653,0.005327,-0.010023,-0.00182,-0.000014,-0.001831,0.006222,0.000567,-0.007013,0.004072,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.000014,-0.001398,0.003713,-0.00056,0.005976,0.000036,-0.71139,2.702128,0.007127,-0.866025,0.5,-2.4493e-16,1.0,0.5,0.866025,0.781831,0.62349,-0.875735,0.482792,-2.4493e-16,1.0,-0.866025,0.5
2016-10-31 03:00:00,0.0,2016,-0.00115,0.013736,-0.018515,0.001733,-0.002529,-0.005653,0.005327,-0.010023,-0.00182,-0.000014,-0.001831,0.006222,0.000567,-0.007013,0.004072,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.000014,-0.001398,-0.000763,0.005907,0.000035,-0.648364,2.863171,0.006892,-0.866025,0.5,-2.4493e-16,1.0,0.707107,0.707107,0.781831,0.62349,-0.875735,0.482792,-2.4493e-16,1.0,-0.866025,0.5
2016-10-31 04:00:00,0.005869,2016,0.0,-0.00115,0.013736,-0.018515,0.001733,-0.002529,-0.005653,0.005327,-0.010023,-0.00182,-0.000014,-0.001831,0.006222,0.000567,-0.007013,0.004072,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.000014,-0.000704,0.005907,0.000035,-0.678312,2.888483,0.006696,-0.866025,0.5,-2.4493e-16,1.0,0.866025,0.5,0.781831,0.62349,-0.875735,0.482792,-2.4493e-16,1.0,-0.866025,0.5
2016-10-31 05:00:00,0.0,2016,0.005869,0.0,-0.00115,0.013736,-0.018515,0.001733,-0.002529,-0.005653,0.005327,-0.010023,-0.00182,-0.000014,-0.001831,0.006222,0.000567,-0.007013,0.004072,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.000459,0.006057,0.000037,-0.696525,2.494112,0.006672,-0.866025,0.5,-2.4493e-16,1.0,0.965926,0.258819,0.781831,0.62349,-0.875735,0.482792,-2.4493e-16,1.0,-0.866025,0.5
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-09-22 18:00:00,-0.002955,2025,0.000284,-0.002176,-0.00163,-0.000909,-0.002023,0.003623,-0.001284,0.000849,0.004273,-0.002853,0.000972,-0.002513,-0.007712,-0.006325,-0.00213,0.003857,-0.000419,-0.001329,-0.007169,-0.002502,0.001105,0.000771,-0.001593,-0.000315,-0.001131,0.003049,0.000009,-0.361024,0.157002,-0.000571,-1.0,-1.8370e-16,-0.968077,-0.250653,-1.0,-1.8370e-16,0.781831,0.62349,-0.99605,-0.088796,-1.0,-1.8370e-16,-0.986731,-0.162366
2025-09-22 19:00:00,0.006117,2025,-0.002955,0.000284,-0.002176,-0.00163,-0.000909,-0.002023,0.003623,-0.001284,0.000849,0.004273,-0.002853,0.000972,-0.002513,-0.007712,-0.006325,-0.00213,0.003857,-0.000419,-0.001329,-0.007169,-0.002502,0.001105,0.000771,-0.001593,-0.001241,0.003065,0.000009,-0.255294,0.048155,-0.000639,-1.0,-1.8370e-16,-0.968077,-0.250653,-0.965926,0.258819,0.781831,0.62349,-0.99605,-0.088796,-1.0,-1.8370e-16,-0.986731,-0.162366
2025-09-22 20:00:00,0.00171,2025,0.006117,-0.002955,0.000284,-0.002176,-0.00163,-0.000909,-0.002023,0.003623,-0.001284,0.00

In [15]:
# Split the test set
split_point = lagged_df.height - (hours ** 2)
train_df = lagged_df.slice(0,split_point)
test_df = lagged_df.slice(split_point, lagged_df.height)
X_test, y_test = test_df.drop('price_pct_change'), test_df['price_pct_change']
X_test.shape, y_test.shape, test_df.shape, train_df.shape, lagged_df.shape

# Split the train and validation sets
split_point_val = int(0.8 * train_df.height)
X, y = train_df.drop('price_pct_change'), train_df['price_pct_change']
X_train = X.slice(0, split_point_val)
y_train = y.slice(0, split_point_val)
X_val = X.slice(split_point_val, train_df.height)
y_val = y.slice(split_point_val, train_df.height)
print(f" X_train: {X_train.shape}\n y_train: {y_train.shape}\n X_val: {X_val.shape}\n y_val: {y_val.shape}\n X_test: {X_test.shape}\n y_test: {y_test.shape}")


 X_train: (61918, 39)
 y_train: (61918,)
 X_val: (15480, 39)
 y_val: (15480,)
 X_test: (576, 39)
 y_test: (576,)


In [16]:
tscv = TimeSeriesSplit(n_splits=5)